# Test Modified Equinotical Orbital Elements

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt
from orbit_utils import *

In [ ]:
t0_tai = pnt.convert_time(pnt.gregorian2time(2025, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

# orbit = "NRHO"
orbit = "POLAR"
# orbit = "DRO"
# orbit = "CLFO"

overwrite = True

if orbit == "ELFO":
    a = 6541.4
    ecc = 0.6
    inc = np.deg2rad(65.5)
    Omega = np.deg2rad(60)
    w = np.deg2rad(90)
    M0 = np.deg2rad(0)

    coe_op = np.array([a, ecc, inc, Omega, w, M0])
    rv0_op = pnt.classical_to_cart(coe_op, pnt.GM_MOON)  # In OP frame
    rv0_ci = pnt.convert_frame(t0_tai, rv0_op, pnt.MOON_OP, pnt.MOON_CI)[
        0
    ]  # In CI frame

    sphm = [25, 25]  # Spherical harmonic model degree and order
    orbit_filename = "rv_prop_elfo.npy"
    twobody = True

elif orbit == "POLAR":
    a = 3870.0
    ecc = 0.001
    inc = np.deg2rad(90)
    Omega = np.deg2rad(0)
    w = np.deg2rad(90)
    M0 = np.deg2rad(10)

    coe_op = np.array([a, ecc, inc, Omega, w, M0])
    rv0_op = pnt.classical_to_cart(coe_op, pnt.GM_MOON)  # In OP frame
    rv0_ci = pnt.convert_frame(t0_tai, rv0_op, pnt.MOON_OP, pnt.MOON_CI)[
        0
    ]  # In CI frame

    sphm = [80, 80]  # Spherical harmonic model degree and order
    orbit_filename = "rv_prop_polar.npy"
    twobody = True


elif orbit == "CLFO":
    a = 6215.0
    ecc = 0.0001
    inc = np.deg2rad(39.23)
    Omega = np.deg2rad(0)
    w = np.deg2rad(90)
    M0 = np.deg2rad(60)

    coe_op = np.array([a, ecc, inc, Omega, w, M0])
    rv0_op = pnt.classical_to_cart(coe_op, pnt.GM_MOON)  # In OP frame
    rv0_ci = pnt.convert_frame(t0_tai, rv0_op, pnt.MOON_OP, pnt.MOON_CI)[
        0
    ]  # In CI frame

    sphm = [80, 80]  # Spherical harmonic model degree and order
    orbit_filename = "rv_prop_clfo.npy"
    twobody = True

elif orbit == "Halo_S25":
    t0_tai = pnt.convert_time(pnt.gregorian2time(2021, 1, 1, 0, 0, 0), pnt.UTC, pnt.TAI)
    rv0_ci = np.array(
        [
            -38279.9298,
            61417.8316,
            -50632.1007,
            0.0895177417,
            0.0538711588,
            -0.00233218533,
        ]
    )
    sphm = [8, 8]
    T_syn = 29.68278536728694 * 86400
    # synodic period
    period = T_syn * 2 / 5
    n_period = 3
    orbit_filename = "rv_prop_halo_S25.npy"
    twobody = False

elif orbit == "NRHO":
    t0_tai = pnt.convert_time(
        pnt.gregorian2time(2022, 7, 15, 0, 0, 0), pnt.UTC, pnt.TAI
    )
    rv0_ci = np.array(
        [
            4811.24915,
            23140.1109,
            -66343.1582,
            -0.0560634369,
            -0.0401916424,
            -0.0180843762,
        ]
    )
    sphm = [8, 8]
    T_syn = 29.68278536728694 * 86400
    # synodic period
    period = T_syn * 2 / 9
    n_period = 13
    orbit_filename = "rv_prop_nrho.npy"
    twobody = False

elif orbit == "DRO":
    # https://dataverse.jpl.nasa.gov/file.xhtml?fileId=58758&version=2.0
    # "Efficient NRHO to DRO transfers in Cislunar Space"
    t0_tai = pnt.convert_time(
        pnt.gregorian2time(2025, 1, 29, 6, 12, 58.37833), pnt.UTC, pnt.TAI
    )
    rv0_ci = np.array(
        [
            7.006297998479835e4,
            5.006352175136821e4,
            0.489708863788025e4,
            0.060533650318577,
            -0.111443777216460,
            -0.010470486200574,
        ]
    )
    coe = pnt.cart_to_classical(rv0_ci, pnt.GM_MOON)
    a = coe[0]
    sphm = [8, 8]
    period = 13.0 * 86400
    n_period = 5
    orbit_filename = "rv_prop_dro.npy"
    twobody = False

# propagation timestep
if twobody:
    period = 2 * np.pi * np.sqrt(a**3 / pnt.GM_MOON)
    dt = 5.0
    n_period = 3
else:
    dt = 60.0

tspan = np.arange(0, n_period * period, dt)
t_tai = t0_tai + tspan
lent = tspan.size

print("Initial position and velocity in CI frame:")
print(rv0_ci)

dynamics

In [ ]:
add_earth = True
add_sun = True

dyn = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-12, reltol=1e-12))
dyn.add_body(pnt.Body.Moon(sphm[0], sphm[1]))
if add_earth:
    dyn.add_body(pnt.Body.Earth())
if add_sun:
    dyn.add_body(pnt.Body.Sun())
dyn.set_frame(pnt.MOON_CI)
dyn.set_time_step(10)  # propagation timestep

In [ ]:
import os

save_dir = "data/test_eqoe"
if not os.path.exists(save_dir):
    os.makedirs(save_dir)
filename = os.path.join(save_dir, orbit_filename)

if os.path.exists(filename) and not overwrite:
    rv_prop_mci = np.load(filename)
else:
    rv_prop_mci = dyn.propagate(rv0_ci, t0_tai, t_tai)
    np.save(filename, rv_prop_mci)

print("Final position and velocity in CI frame:")
print(rv_prop_mci[-1])

oe_prop = np.zeros(rv_prop_mci.shape)
for i, rv in enumerate(rv_prop_mci):
    oe_prop[i] = pnt.cart_to_classical(rv, pnt.GM_MOON)

In [ ]:
# 3d plot of the orbit
from plotly import graph_objects as go

# convert to moon fixed frame
rv_prop_bf = convert_ci2pa(t_tai, rv_prop_mci, rotate_only=True)

fig = go.Figure()
if orbit == "DRO":
    rv_prop_eci = rv_prop_mci + pnt.get_body_pos_vel(
        t_tai, pnt.EARTH, pnt.MOON, pnt.Frame.ECI
    )  # (S - M) + (M - E) = S - E
    lunar_orbit_eci = pnt.get_body_pos_vel(t_tai, pnt.MOON, pnt.Frame.ECI)

    print(lunar_orbit_eci.shape)

    lunar_orbit_pa = convert_ci2pa(t_tai, lunar_orbit_eci, rotate_only=True)
    rv_prop_pa = convert_ci2pa(t_tai, rv_prop_eci, rotate_only=True)

    orbits = np.zeros((2, t_tai.size, 6))
    orbits[0] = lunar_orbit_eci
    orbits[1] = rv_prop_eci
    pnt.plot.plot_orbits(fig, orbits, color=["gray", "blue"])

elif orbit == "NRHO" or orbit == "Halo_S25":
    pnt.plot.plot_orbits(fig, rv_prop_bf, color="blue")
    pnt.plot.plot_body(
        fig,
        pnt.MOON,
        size_factor=2,
        alpha=0.5,
    )
else:
    pnt.plot.plot_orbits(fig, rv_prop_mci, color="blue")
    pnt.plot.plot_body(
        fig,
        pnt.MOON,
        size_factor=2,
        alpha=0.5,
    )

pnt.plot.set_view(fig, -80, 20, 2.5)
fig.update_layout(showlegend=True, width=400, height=400)
fig.show()

implementation tests

In [ ]:
from orbit_utils import cart_to_mqoe, mqoe_to_cart

# compute modified equinoctial elements
mqoe_from_cart = cart_to_mqoe(rv_prop_mci, pnt.GM_MOON)
mqoe_from_cart1 = cart_to_mqoe(rv_prop_mci[0], pnt.GM_MOON)

print(mqoe_from_cart.shape)
print(mqoe_from_cart1.shape)

print(mqoe_from_cart[0] == mqoe_from_cart1)

# convert back to Cartesian
cart_from_mqoe = mqoe_to_cart(mqoe_from_cart, pnt.GM_MOON)
cart_from_mqoe1 = mqoe_to_cart(mqoe_from_cart1, pnt.GM_MOON)

print(cart_from_mqoe.shape)
print(cart_from_mqoe1.shape)

print(np.allclose(cart_from_mqoe, rv_prop_mci))
print(np.allclose(cart_from_mqoe1, rv_prop_mci[0]))

In [ ]:
from copy import deepcopy


def plot_oe(tspan, n_period, period, type="mqoe", M_peri=0, M_apo=0, from_mqoe=False):
    # plot the modified equinoctial elements
    fig, axs = plt.subplots(2, 3, figsize=(12, 8))

    if type == "mqoe":
        oe_labels = ["p", "f", "g", "h", "k", "L"]
        scale = np.array([1, 1, 1, 1, 1, 180 / np.pi])
        oe = deepcopy(mqoe_from_cart)
    else:
        oe_labels = [
            "a",
            "e",
            "i [deg]",
            "$\\Omega$ [deg]",
            "$\\omega$ [deg]",
            "M [deg]",
        ]
        scale = np.array([1, 1, 180 / np.pi, 180 / np.pi, 180 / np.pi, 180 / np.pi])
        if from_mqoe:
            oe = mqoe_to_coe(mqoe_from_cart)
        else:
            oe = deepcopy(oe_prop)

    for i in range(6):
        oe[:, i] = scale[i] * oe[:, i]
        # oe[:, i] = np.unwrap(oe[:, i], 180)

    fig.suptitle("Modified Equinoctial Elements (3 periods)")
    for i, ax in enumerate(axs.flatten()):
        ax.plot(tspan / 3600, oe[:, i])
        ax.set_title(f"{oe_labels[i]}")
        ax.set_xlabel("Time [hr]")
        ax.set_ylabel(oe_labels[i])
        ax.grid()

        for k in range(n_period):
            # plot the vertical line
            ax.axvline((k + 1) * period / 3600, color="k", linestyle="--")

    plt.tight_layout()

    # plot near the periapsis
    twidth = 60 * 60  # 30 minute
    peri_t = ((M_peri + 360) / 360) * period
    time_peri = [peri_t - twidth, peri_t + twidth]
    idx = np.where((tspan >= time_peri[0]) & (tspan <= time_peri[1]))[0]

    fig, axs = plt.subplots(2, 3, figsize=(12, 8))
    plt.suptitle("Modified Equinoctial Elements near M={}".format(M_peri))

    for i, ax in enumerate(axs.flatten()):
        ax.plot(tspan[idx] / 60, oe[idx, i])
        ax.set_title(f"{oe_labels[i]}")
        ax.set_xlabel("Time [min]")
        ax.grid()

    plt.tight_layout()

    # plot the modified equinoctial elements
    fig, axs = plt.subplots(2, 3, figsize=(12, 8))
    plt.suptitle("Modified Equinoctial Elements near M={}".format(M_apo))
    apo_t = ((M_apo + 180) / 360) * period
    time_apo = [apo_t - twidth, apo_t + twidth]
    idx = np.where((tspan >= time_apo[0]) & (tspan <= time_apo[1]))[0]

    for i, ax in enumerate(axs.flatten()):
        ax.plot(tspan[idx] / 60, oe[idx, i])
        ax.set_title(f"{oe_labels[i]}")
        ax.set_xlabel("Time [min]")
        ax.grid()

    plt.tight_layout()

In [ ]:
plot_oe(tspan, n_period, period, type="coe", from_mqoe=False)

In [ ]:
plot_oe(tspan, n_period, period, type="mqoe")